# Phase 3 - Feature Engineering

This notebook demonstrates the complete Phase 3 feature engineering process as dictated by the project architecture.

## Workflow:
1. Load Aligned Phase 2 Datasets
2. NIFTY 50 Benchmark Features
3. Sector Features
4. Volume Inspection (Result: Excluded)
5. Final Feature Compilation & Validation
6. Save Feature-Engineered Datasets

In [1]:
import pandas as pd
import numpy as np
import os
import sys
from pathlib import Path

# Add project root to sys path
project_root = Path(os.getcwd()).parent
sys.path.append(str(project_root))

from src.feature_engineering import calculate_returns, calculate_momentum_sma, calculate_relative_strength, calculate_volatility, calculate_market_trend

In [2]:
### 1. Load Aligned Phase 2 Datasets
aligned_dir = project_root / 'data' / 'processed' / 'aligned'
output_dir = project_root / 'data' / 'processed' / 'feature_engineered'

# Ensure output directory exists
output_dir.mkdir(parents=True, exist_ok=True)

sector_files = {
    'AUTO': 'nifty_auto_aligned.csv',
    'BANK': 'nifty_bank_aligned.csv',
    'FMCG': 'nifty_fmcg_aligned.csv',
    'IT': 'nifty_it_aligned.csv',
    'METAL': 'nifty_metal_aligned.csv',
    'PHARMA': 'nifty_pharma_aligned.csv'
}

# Load NIFTY 50 (Benchmark)
nifty50_path = aligned_dir / 'nifty50_aligned.csv'
df_nifty = pd.read_csv(nifty50_path, index_col='Date', parse_dates=True)

# Load all sectors
sectors_data = {name: pd.read_csv(aligned_dir / filename, index_col='Date', parse_dates=True) 
                for name, filename in sector_files.items()}

### Volume Reliability Inspection
During preliminary inspection of the Phase 2 dataset, it was found that the Volume column contained substantial zero values across core indices (e.g., NIFTY Bank and NIFTY IT had over 420 zeros out of 1886 rows, roughly 22%).

**Decision:** Volume Participation has been EXCLUDED from feature engineering to prevent mathematical anomalies and divide-by-zero errors. No Volume features will be generated.

In [3]:
### 2. Calculate NIFTY 50 Benchmark Features
df_nifty_features = df_nifty.copy()

# Returns
df_nifty_features = calculate_returns(df_nifty_features, column='Close', windows=[20, 60])
# Momentum (20D, 60D)
df_nifty_features = calculate_momentum_sma(df_nifty_features, column='Close', windows=[20, 60], prefix='NIFTY50_Momentum')
# Volatility (20D Only for NIFTY 50)
df_nifty_features = calculate_volatility(df_nifty_features, column='Close', windows=[20])
# Market Trend (200D Only for NIFTY 50)
df_nifty_features = calculate_market_trend(df_nifty_features, column='Close', window=200, col_name='NIFTY50_Market_Trend_200D')

print("NIFTY 50 Benchmark Features Generated.")

NIFTY 50 Benchmark Features Generated.


In [4]:
### 3. Calculate Sector Features
sectors_features = {}

for name, df_sector in sectors_data.items():
    feat_df = df_sector.copy()
    
    # Returns (20D, 60D)
    feat_df = calculate_returns(feat_df, column='Close', windows=[20, 60])
    
    # Relative Strength against NIFTY 50 (20D, 60D)
    feat_df = calculate_relative_strength(feat_df, df_nifty_features, windows=[20, 60])
    
    # Volatility for Sector (20D, 60D)
    feat_df = calculate_volatility(feat_df, column='Close', windows=[20, 60])
    
    sectors_features[name] = feat_df

print("Sector Features Generated for all 6 indices.")

Sector Features Generated for all 6 indices.


In [5]:
### 4. Comprehensive Validation
def run_validations():
    print("--- Starting Validation ---\n")
    
    # A. Feature Existence
    expected_nifty = ['Return_20D', 'Return_60D', 'NIFTY50_Momentum_20D', 'NIFTY50_Momentum_60D', 'Volatility_20D', 'NIFTY50_Market_Trend_200D']
    expected_sector = ['Return_20D', 'Return_60D', 'Relative_Strength_20D', 'Relative_Strength_60D', 'Volatility_20D', 'Volatility_60D']
    
    missing_nifty = [f for f in expected_nifty if f not in df_nifty_features.columns]
    assert len(missing_nifty) == 0, f"Missing NIFTY features: {missing_nifty}"
    
    for name, df in sectors_features.items():
        missing_sec = [f for f in expected_sector if f not in df.columns]
        assert len(missing_sec) == 0, f"Missing {name} features: {missing_sec}"
    print("✅ Feature Existence Verified")
        
    # B. Date/Index Integrity
    assert df_nifty_features.index.is_monotonic_increasing, "NIFTY Index not strictly increasing"
    assert not df_nifty_features.index.has_duplicates, "NIFTY Index has duplicates"
    for name, df in sectors_features.items():
        assert df.index.equals(df_nifty_features.index), f"{name} index does not perfectly align with NIFTY 50"
    print("✅ Date/Index Integrity Verified")
    
    # C. NaN Validation
    # 20D should have exactly 20 initial NaNs (indices 0 to 19)
    assert df_nifty_features['Return_20D'].isna().sum() == 20
    # 60D should have exactly 60 initial NaNs
    assert df_nifty_features['Return_60D'].isna().sum() == 60
    # 200D Trend should have exactly 199 initial NaNs (SMA requires 200 items, so 0 to 198 are NaN, 199 is first valid)
    assert df_nifty_features['NIFTY50_Market_Trend_200D'].isna().sum() == 199
    print("✅ NaN Boundary Integrity Verified")
    
    # D. Reasonableness
    assert not np.isinf(df_nifty_features['Return_20D']).any(), "Found infinite values"
    print("✅ Values Reasonableness Verified")
    print("\nAll automated validations PASSED.")
    
run_validations()

--- Starting Validation ---

✅ Feature Existence Verified
✅ Date/Index Integrity Verified
✅ NaN Boundary Integrity Verified
✅ Values Reasonableness Verified

All automated validations PASSED.


In [6]:
### 5. Save Feature-Engineered Datasets
df_nifty_features.to_csv(output_dir / 'nifty50_features.csv')
print("Saved NIFTY 50 Features")

for name, df in sectors_features.items():
    df.to_csv(output_dir / f'nifty_{name.lower()}_features.csv')
    print(f"Saved {name} Features")

print("\nPhase 3 Feature Engineering Complete. All datasets saved successfully.")

Saved NIFTY 50 Features
Saved AUTO Features
Saved BANK Features
Saved FMCG Features
Saved IT Features
Saved METAL Features
Saved PHARMA Features

Phase 3 Feature Engineering Complete. All datasets saved successfully.
